# Lakehouse Agent — Two-Pattern Fine-Grained Access Control (Streamlit demo)

Test the full agent end-to-end and **see** fine-grained access control at work: per-role **tool gating** and per-identity **data scope**, across two identity-propagation patterns (claims gateway **GW1** request/response interceptors · notes gateway **GW2** per-user notes isolation). Log in as different personas and watch what each is allowed to see change with *who they are* — their token is used invisibly, never shown.

## Prerequisites

- ✅ Run `06-deploy-agent.ipynb` first
- ✅ All components deployed

## What This Notebook Does

This notebook adapts to the active `IDP_PROVIDER` (chosen in `01-deploy-idp.ipynb`):

- **[COGNITO]** Steps 1–2 run a quick notebook token test (client-credentials) + an agent invocation.
- **[OKTA]** Auth is browser-interactive, so there is no notebook token test — skip to Step 3 and log in via Okta in the app.

Both paths then launch the Streamlit UI (Step 3) for interactive testing. Step 3 starts the app **non-blocking** with `LAKEHOUSE_SAVE_TOKENS=1`, so each persona login saves that persona's token to `.tmp/<persona>_token.txt`. The **Fine-grained access control** cells that follow read those tokens and prove per-role tool gating + per-identity data scope deterministically (identical on both IdP paths).

## Important Notes

⚠️ **Run cells in order**: Start with the Setup cell (cell 2) to initialize the AWS session, clients, and read the IdP flag.

In [ ]:
# ============================================================================
# SETUP CELL - Run this first to initialize AWS session and clients
# ============================================================================

# AWS Initialization - Load credentials and create session
from utils.notebook_init import init_aws
from utils.idp_config import get_idp_provider
import json
import base64
import requests
import uuid
import urllib.parse

# This will:
# 1. Load credentials from .env file (if it exists)
# 2. Create and validate AWS session (env vars take precedence over SSO)
# 3. Return session, region, and account_id for use in this notebook
session, region, account_id = init_aws()

# Initialize AWS clients
ssm_client = session.client("ssm", region_name=region)

# Read the IdP flag once (set in 01-deploy-idp.ipynb Step 0). Downstream cells
# branch on this to decide whether to run the Cognito-only notebook token test.
IDP_PROVIDER = get_idp_provider(ssm_client)

print("✅ Ready to proceed with AWS operations")
print(f"   Account ID: {account_id}")
print(f"   Region: {region}")
print(f"   IdP Provider: {IDP_PROVIDER}")
print("\n📝 Architecture: User → Agent Runtime → Gateway → MCP Server")

## [COGNITO] Step 1: Get OAuth Token from Cognito

**Cognito only.** This client-credentials token test is skipped on the Okta path (browser-interactive auth — see Step 3).

In [ ]:
# [COGNITO] Client-credentials token test — skipped on the Okta path.
if IDP_PROVIDER == "cognito":
    # Get Cognito configuration from SSM
    COGNITO_DOMAIN = ssm_client.get_parameter(Name="/app/lakehouse-agent/cognito-domain")["Parameter"]["Value"]

    CLIENT_ID = ssm_client.get_parameter(Name="/app/lakehouse-agent/cognito-app-client-id")["Parameter"]["Value"]

    CLIENT_SECRET = ssm_client.get_parameter(
        Name="/app/lakehouse-agent/cognito-app-client-secret", WithDecryption=True
    )["Parameter"]["Value"]

    print("🔐 Cognito Configuration:")
    print(f"   Domain: {COGNITO_DOMAIN}")
    print(f"   Client ID: {CLIENT_ID}")

    # Request token
    token_url = f"{COGNITO_DOMAIN}/oauth2/token"
    credentials = f"{CLIENT_ID}:{CLIENT_SECRET}"
    encoded_credentials = base64.b64encode(credentials.encode()).decode()

    headers = {
        "Authorization": f"Basic {encoded_credentials}",
        "Content-Type": "application/x-www-form-urlencoded",
    }

    data = {"grant_type": "client_credentials", "scope": "lakehouse-api/claims.query"}

    print("\n🔑 Requesting OAuth token...")
    response = requests.post(token_url, headers=headers, data=data)

    if response.status_code == 200:
        token_data = response.json()
        ACCESS_TOKEN = token_data["access_token"]
        print("✅ OAuth token obtained successfully!")
        print(f"   Token type: {token_data.get('token_type')}")
        print(f"   Expires in: {token_data.get('expires_in')} seconds")
    else:
        print(f"❌ Failed to get token: {response.status_code}")
        print(response.text)
        ACCESS_TOKEN = None
else:
    # [OKTA] Browser-interactive auth — no notebook token test.
    ACCESS_TOKEN = None
    print("ℹ️  [OKTA] Okta auth is browser-interactive — no notebook token test.")
    print("    Launch the app (Step 3) and log in via Okta.")

## [COGNITO] Step 2: Test Agent Invocation

**Cognito only** (uses the client-credentials token from Step 1). Skipped on the Okta path.

**Architecture Flow:**
1. User → Agent Runtime (OAuth token in Authorization header for JWT validation)
2. Agent receives token in payload (JWT authorizer consumes header, doesn't pass through)
3. Agent → Gateway (passes token from payload)
4. Gateway → MCP Server (with user context)

**Note:** The bearer token must be passed in BOTH the Authorization header (for JWT validation) AND the payload (for the agent code to use when calling Gateway). This is because the JWT authorizer consumes the Authorization header and doesn't pass it through to the agent code.

In [ ]:
# [COGNITO] Agent invocation test — skipped on the Okta path.
if IDP_PROVIDER == "cognito":
    if ACCESS_TOKEN:
        # Get Agent Runtime ARN from SSM
        try:
            AGENT_RUNTIME_ARN = ssm_client.get_parameter(Name="/app/lakehouse-agent/agent-runtime-arn")[
                "Parameter"
            ]["Value"]

            print("🤖 Agent Runtime Configuration:")
            print(f"   Runtime ARN: {AGENT_RUNTIME_ARN}")
            print(f"   Region: {region}")
        except ssm_client.exceptions.ParameterNotFound:
            print("❌ Agent Runtime ARN not found in SSM")
            print("   Please run 06-deploy-agent.ipynb first")
            AGENT_RUNTIME_ARN = None

        if AGENT_RUNTIME_ARN:
            # Construct the AgentCore Runtime invocation URL
            # URL encode the agent ARN
            escaped_agent_arn = urllib.parse.quote(AGENT_RUNTIME_ARN, safe="")
            AGENT_RUNTIME_URL = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

            print(f"   Runtime URL: {AGENT_RUNTIME_URL}")

            # Generate session ID for this invocation
            session_id = f"test-session-{uuid.uuid4()}"

            # Prepare payload with bearer token for Gateway calls
            # Note: Token must be in BOTH header (for JWT auth) and payload (for agent to use)
            payload = {
                "prompt": "Show me all my claims",
                "bearer_token": ACCESS_TOKEN,  # Pass token in payload for agent to use with Gateway
            }

            # Prepare headers with OAuth token and session ID
            headers = {
                "Authorization": f"Bearer {ACCESS_TOKEN}",
                "Content-Type": "application/json",
                "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
            }

            print("\n🚀 Invoking Agent Runtime...")
            print(f"   Prompt: {payload['prompt']}")
            print(f"   Session ID: {session_id}")
            print("   Auth: Bearer token in header (for JWT validation) and payload (for Gateway)")

            try:
                # Call the agent runtime
                response = requests.post(AGENT_RUNTIME_URL, headers=headers, data=json.dumps(payload), timeout=60)

                print(f"\n📊 Response Status: {response.status_code}")

                if response.status_code == 200:
                    try:
                        result = response.json()
                        print("\n✅ Agent Response:")
                        print(json.dumps(result, indent=2))

                        # Display the content if available
                        if "content" in result:
                            print("\n📝 Agent Output:")
                            print(result["content"])

                        if "tool_calls" in result:
                            print(f"\n🔧 Tool Calls: {result['tool_calls']}")

                    except json.JSONDecodeError:
                        print(f"Response: {response.text[:500]}")

                elif response.status_code == 401:
                    print("❌ Unauthorized - OAuth token validation failed")
                    print("   Check that:")
                    print("      1. Agent Runtime has JWT authorizer configured")
                    print("      2. Client ID matches the allowed clients")
                    print("      3. Token has not expired")
                    print(f"\n   Response: {response.text[:500]}")

                elif response.status_code == 403:
                    print("❌ Forbidden - User not authorized")
                    print(f"   Response: {response.text[:500]}")

                elif response.status_code == 424:
                    print("❌ Failed Dependency - Runtime returned 500 error")
                    print(f"   Response: {response.text[:500]}")
                    print("\n   This means the agent code is crashing.")
                    print("   Common causes:")
                    print("      1. Missing Gateway ARN in SSM (/app/lakehouse-agent/gateway-arn)")
                    print("      2. Agent runtime IAM role lacks SSM permissions")
                    print("      3. Agent runtime IAM role lacks bedrock-agentcore-control:GetGateway permission")
                    print("      4. Bearer token not being passed correctly")
                    print("\n   👉 Run the cells below to diagnose:")
                    print('      - "Verify Configuration" cell to check SSM parameters')
                    print('      - "Check CloudWatch Logs" cell to see agent error logs')

                else:
                    print("❌ Request failed")
                    print(f"   Response: {response.text[:500]}")

            except requests.exceptions.Timeout:
                print("\n❌ Request timed out after 60 seconds")
                print("   Check CloudWatch logs:")
                print(f"      - Agent Runtime: /aws/bedrock-agentcore/runtime/{AGENT_RUNTIME_ARN.split('/')[-1]}")
                print("      - Gateway Interceptor: /aws/lambda/lakehouse-gateway-interceptor")

            except Exception as e:
                print(f"\n❌ Error: {e}")
                import traceback

                traceback.print_exc()
    else:
        print("⚠️  Skipping agent test - no access token")
else:
    print("ℹ️  [OKTA] Browser-interactive auth — skipping the notebook invoke test; use Step 3.")

## Step 3: Launch Streamlit UI (and capture persona tokens)

Launch the interactive Streamlit UI for conversational testing with the agent. **This step is IdP-agnostic** — the app reads `IDP_PROVIDER` and shows the matching login (Cognito password form or Okta browser login).

The launch is **non-blocking** (`subprocess.Popen`, not `subprocess.run`), so the kernel stays free for the fine-grained access-control cells below. It sets `LAKEHOUSE_SAVE_TOKENS=1`, which tells `streamlit-ui/streamlit_app.py` to persist each login's access token to `.tmp/<persona>_token.txt` (`<persona>` = the email local part, file mode `0600`, value never rendered — see `_lakehouse_save_token_if_enabled`).

**Log in as each persona in the browser** — every login writes that persona's token file:

| Persona login | Token file |
|---|---|
| `policyholder001@example.com` | `.tmp/policyholder001_token.txt` |
| `policyholder002@example.com` | `.tmp/policyholder002_token.txt` |
| `admin@example.com` | `.tmp/admin_token.txt` |

On **Cognito** the login is the sidebar password form (test users use password `TempPass123!`); on **Okta** it is the browser Authorization-Code + PKCE flow. Keep the app running while you work through the FGAC cells, then run the **Stop Streamlit** cell at the end.

> Access tokens are short-lived (~60 min) — if the FGAC cells start failing on auth, log the persona in again to refresh its file.

In [ ]:
import os
import subprocess

print("🚀 Launching Streamlit UI (non-blocking)...")
print("\n📝 Instructions:")
print("   - Streamlit opens in your browser automatically")
print("   - Log in via the sidebar (Cognito password form or Okta browser login, per IDP_PROVIDER)")
print("   - Each login saves that persona's token to .tmp/<persona>_token.txt")
print("     (LAKEHOUSE_SAVE_TOKENS=1); log in as all three to run the FGAC cells:")
print("       policyholder001@example.com")
print("       policyholder002@example.com")
print("       admin@example.com")
print('   - Try queries like: "Show me all claims" or "Get claims summary"')
print('   - Run the "Stop Streamlit" cell at the end when you are done')

# Popen (not run) so the notebook kernel is NOT blocked while Streamlit serves.
# LAKEHOUSE_SAVE_TOKENS=1 tells streamlit_app.py to persist each persona's access
# token to .tmp/<persona>_token.txt on login, so the FGAC cells below can read them.
try:
    streamlit_dir = os.path.join(os.getcwd(), "streamlit-ui")
    streamlit_proc = subprocess.Popen(
        ["streamlit", "run", "streamlit_app.py"],
        cwd=streamlit_dir,
        env={**os.environ, "LAKEHOUSE_SAVE_TOKENS": "1"},
    )
    print(f"\n✅ Streamlit started (PID {streamlit_proc.pid}) — kernel is free.")
except FileNotFoundError:
    print("\n❌ streamlit-ui directory, streamlit_app.py, or the streamlit CLI not found")
    print("   Make sure you are running this from the lakehouse-agent directory")
except Exception as e:
    print(f"\n❌ Error launching Streamlit: {e}")
    print("\n💡 Manual launch:")
    print("   cd streamlit-ui")
    print("   LAKEHOUSE_SAVE_TOKENS=1 streamlit run streamlit_app.py")

## Fine-grained access control, made visible

We surface FGAC two complementary ways — run the two cells below, then read the interpretation that follows:

1. **Which tools each role sees** — the RESPONSE interceptor filters the claims gateway's (GW1) `tools/list` by the caller's group claim (`cognito:groups` on Cognito, `groups` on Okta), so different personas are offered different tools.
2. **What data each identity sees** — the same tool returns only *your* rows; admin's deliberately unscoped SQL tool sees the whole book.

Both cells read the persona tokens captured by the Streamlit logins above (`.tmp/<persona>_token.txt`) and call GW1 directly, so they are **IdP-agnostic** — they run identically on the Cognito and Okta paths.

In [ ]:
# ── Tool gating, made visible: what each persona's toolset looks like [FGAC-TOOLSETS] ──
# The RESPONSE interceptor filters the claims gateway's (GW1) tools/list by the caller's
# group claim — same gateway, same request, different result per identity.
import asyncio
import os

import nest_asyncio
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Jupyter already runs an event loop; nest_asyncio lets asyncio.run work inside it.
nest_asyncio.apply()

# Canonical GW1 (claims gateway) SSM key — written by deployment/5a-gateway-setup/create_gateway.py
# and read the same way by streamlit-ui/streamlit_app.py.
CLAIMS_GATEWAY_URL = ssm_client.get_parameter(Name="/app/lakehouse-agent/gateway-url")["Parameter"]["Value"]


def _find_tmp_dir(start=None):
    # .tmp lives at the workspace root, OUTSIDE the git repo; walk up to find it.
    d = os.path.abspath(start or os.getcwd())
    while True:
        cand = os.path.join(d, ".tmp")
        if os.path.isdir(cand):
            return cand
        parent = os.path.dirname(d)
        if parent == d:
            raise FileNotFoundError("Could not locate a .tmp directory walking up from cwd")
        d = parent


TMP_DIR = _find_tmp_dir()


def _load_token(persona):
    """Read .tmp/<persona>_token.txt, written by the Streamlit login (LAKEHOUSE_SAVE_TOKENS=1)."""
    path = os.path.join(TMP_DIR, f"{persona}_token.txt")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — launch the app (Step 3) and log in as {persona} first.")
    with open(path) as f:
        return f.read().strip()


async def _list_tool_names(token):
    headers = {"Authorization": f"Bearer {token}"}
    async with (
        streamablehttp_client(CLAIMS_GATEWAY_URL, headers=headers) as (read, write, _),
        ClientSession(read, write) as sess,
    ):
        await sess.initialize()
        resp = await sess.list_tools()
        # bare tool names (strip the "<target>___" gateway prefix)
        return sorted({t.name.split("___", 1)[1] if "___" in t.name else t.name for t in resp.tools})


PERSONAS = {
    "policyholder001 (John Doe)": "policyholder001",
    "admin (administrators)": "admin",
}

print("🧰 Tools each persona can see on the claims gateway / GW1 (tools/list):\n")
persona_toolsets = {}
for label, persona in PERSONAS.items():
    names = asyncio.run(_list_tool_names(_load_token(persona)))
    persona_toolsets[label] = names
    print(f"  {label:28s} ({len(names)} tools): {names}")

SEARCH_TOOL = "x_amz_bedrock_agentcore_search"
pol_set = set(persona_toolsets["policyholder001 (John Doe)"])
adm_set = set(persona_toolsets["admin (administrators)"])

# allowed_tools per group come from get_seed_data() in
# deployment/5a-gateway-setup/interceptor-request/setup_dynamodb_tenant_role_maps.py:
#   policyholders  -> get_claims_summary, get_claim_details, query_claims
#   administrators -> query_login_audit, text_to_sql
audit_admin_only = "query_login_audit" in adm_set and "query_login_audit" not in pol_set
sql_admin_only = "text_to_sql" in adm_set and "text_to_sql" not in pol_set
search_stripped = all(SEARCH_TOOL not in names for names in persona_toolsets.values())

print("\n— checks —")
print(f"  toolsets differ ................. {pol_set != adm_set}")
print(f"  query_login_audit admin-only .... {audit_admin_only}")
print(f"  text_to_sql admin-only .......... {sql_admin_only}")
print(f"  system search tool stripped ..... {search_stripped}")

In [ ]:
# ── Row-level data scope, made visible: same tool, different identity [FGAC-DATA-CONTRAST] ──
# John Doe and Jane Smith call the SAME tool (get_claims_summary) and each sees only THEIR
# rows — the identity propagated by the REQUEST interceptor becomes the SQL predicate.
# Admin cannot call that policyholder tool at all (tool ACCESS is gated too), so the
# full-book number comes from admin's own tool, text_to_sql: intentionally unscoped,
# because the administrators role holds the full-table Lake Formation grant.
import json


async def _call_named_tool(token, bare_suffix, arguments=None):
    headers = {"Authorization": f"Bearer {token}"}
    async with (
        streamablehttp_client(CLAIMS_GATEWAY_URL, headers=headers) as (read, write, _),
        ClientSession(read, write) as sess,
    ):
        await sess.initialize()
        tools = await sess.list_tools()
        target = next((t.name for t in tools.tools if t.name.endswith(bare_suffix)), None)
        if target is None:
            seen = [t.name for t in tools.tools]
            raise RuntimeError(f"{bare_suffix} not advertised to this persona; saw {seen}")
        res = await sess.call_tool(target, arguments=arguments or {})
        payloads = []
        for chunk in getattr(res, "content", []) or []:
            txt = getattr(chunk, "text", None)
            if not txt:
                continue
            try:
                payloads.append(json.loads(txt))
            except ValueError:
                payloads.append({"_raw": txt})
        return payloads


def _as_int(value):
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return None


def _walk_for_total_claims(obj):
    """Depth-first search for a total_claims value in a nested payload."""
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key.lower() == "total_claims":
                found = _as_int(value)
                if found is not None:
                    return found
        for value in obj.values():
            found = _walk_for_total_claims(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _walk_for_total_claims(value)
            if found is not None:
                return found
    return None


def _find_total_claims(payloads):
    """Pull a claim count out of either live shape.

    get_claims_summary -> summary.total_claims (int)
    text_to_sql        -> results[0].total_claims (Athena returns strings)
    """
    for payload in payloads:
        found = _walk_for_total_claims(payload)
        if found is not None:
            return found
    # Fallback for text_to_sql if the generated SQL aliased the count column
    # differently: a single-row, single-column result IS the count.
    for payload in payloads:
        rows = payload.get("results") if isinstance(payload, dict) else None
        if isinstance(rows, list) and len(rows) == 1 and isinstance(rows[0], dict) and len(rows[0]) == 1:
            return _as_int(next(iter(rows[0].values())))
    return None


ADMIN_NL_QUERY = "How many claims are in the claims table in total? Return one column aliased total_claims."

# (label, persona token file, a tool the caller is actually allowed to use, arguments)
CONTRAST = [
    ("policyholder001 (John Doe)", "policyholder001", "get_claims_summary", {}),
    ("policyholder002 (Jane Smith)", "policyholder002", "get_claims_summary", {}),
    ("admin (full book)", "admin", "text_to_sql", {"natural_language_query": ADMIN_NL_QUERY}),
]

print("🔎 Same question, different identity, different rows:\n")
counts = {}
for label, persona, tool, arguments in CONTRAST:
    payloads = asyncio.run(_call_named_tool(_load_token(persona), tool, arguments))
    counts[label] = _find_total_claims(payloads)
    print(f"  {label:28s} via {tool:19s} → total_claims = {counts[label]}")
    for payload in payloads:
        if isinstance(payload, dict) and payload.get("generated_sql"):
            sql = " ".join(payload["generated_sql"].split())
            print(f"      generated SQL (admin, unscoped): {sql}")

john = counts["policyholder001 (John Doe)"]
jane = counts["policyholder002 (Jane Smith)"]
admin = counts["admin (full book)"]
halves = None if john is None or jane is None else john + jane
tie = halves is not None and admin is not None and halves == admin
print(f"\n  John Doe({john}) + Jane Smith({jane}) = {halves}   vs   admin full-book({admin})")
print(f"  tie-out {'✅' if tie else '❌'} — the two private views reconstitute the full book, nothing extra")

<!-- NARRATIVE-INTERP -->
### What those two cells show

**Tool gating (per role).** John Doe (policyholder) is offered 3 tools — `get_claims_summary`, `get_claim_details`, `query_claims`; admin is offered 2 — `query_login_audit`, `text_to_sql`. Both admin tools are admin-only, and the `x_amz_bedrock_agentcore_search` system tool is stripped from both sets. Gating is **DynamoDB-driven** (`allowed_tools` per group in `lakehouse_tenant_role_map`, seeded by `get_seed_data()` in `deployment/5a-gateway-setup/interceptor-request/setup_dynamodb_tenant_role_maps.py`) and enforced **two ways**: the RESPONSE interceptor filters `tools/list` (visibility), and the REQUEST interceptor returns **403** on a disallowed call (enforcement) — a hidden tool cannot be called even if its name is known. Notebook `07` asserts both halves.

*Least privilege, not an inverted hierarchy:* admin's two tools are the broad, powerful ones (arbitrary SQL over the full table + the login-audit read), which subsume the scoped reads; policyholders get narrow, self-scoping self-service tools. Each role gets exactly what its job needs.

**Data scope (per identity).** Same tool, different caller: John Doe's `get_claims_summary` returns **4** claims, Jane Smith's returns **5** — each sees only their own. Admin has no access to that policyholder tool, so the full-book figure comes from admin's own `text_to_sql`, which counts **9**. **4 + 5 = 9**: the two private views exactly reconstitute the full portfolio (`CLM-2024-001…004` are John's, `CLM-2024-005…009` are Jane's — see `deployment/3-s3tables-setup/load_sample_data.py`), with no cross-user leakage and no rows unaccounted for.

**How the row-scoping actually works.** The REQUEST interceptor validates the caller's JWT and propagates their identity (email) to the MCP server. The scoped claims tools filter rows with a SQL predicate bound to that identity (`WHERE user_id = ?`), so a caller only ever queries their own rows. **Lake Formation** governs the table itself — per-role table grants + column filtering (administrators hold the full-table grant including the PII columns, which is what lets the deliberately unscoped `text_to_sql` aggregate all 9). In short: **row scope = identity-propagated SQL predicate; Lake Formation = table/column governance**, not a per-user LF row filter for these tools.

### Stop Streamlit

Run this when you are done capturing tokens / chatting with the agent — it terminates the server started in Step 3.

In [ ]:
# Stop the Streamlit server launched above (frees the process + port).
try:
    streamlit_proc.terminate()
    streamlit_proc.wait(timeout=10)
    print("✅ Streamlit stopped")
except NameError:
    print("ℹ️ No Streamlit handle — run the launch cell above first.")
except Exception as e:
    print(f"⚠️ Could not stop Streamlit cleanly: {e}")

## Summary

✅ **Testing Complete!**

**Two identity-propagation patterns, one agent:**

```
  User (Cognito or Okta JWT)
        │
        ▼
  Lakehouse Agent (AgentCore Runtime)
        ├──────────────────────────────┬──────────────────────────────
        │ claims/* → GW1               │ notes/* → GW2
        │ REQUEST + RESPONSE           │ [OKTA] OBO token exchange (RFC 8693)
        │ interceptors                 │ [COGNITO] notes REQUEST interceptor
        ▼                              ▼
  Claims gateway (GW1)           Notes gateway (GW2)
   → Claims MCP → Athena/S3 Tables  → OpenSearch MCP (claim notes)
   • per-role tool gating           • per-user record isolation
     (allowed_tools in DynamoDB)      (owner_user_sub)
   • per-identity row scope
     + Lake Formation column filtering
```

**Demo walkthrough** — log in as each persona and watch access change with identity:

- **John Doe / Jane Smith (policyholders):** 3 self-service claims tools; each sees only their own claims (**4** / **5**).
- **admin:** 2 broad tools (`query_login_audit`, `text_to_sql`); `text_to_sql` sees the full book (**9**).
- Ask *"what tools do you have access to?"* per persona to hear the same gating from the agent itself.

### [COGNITO]

- Notebook token test (client-credentials) + agent invocation validated above.
- **Test users** (password `TempPass123!`):
  - policyholder001@example.com / policyholder002@example.com
  - adjuster001@example.com / adjuster002@example.com / admin@example.com
- User identity is injected by the REQUEST interceptor (look for `X-User-Principal` / `context.user_id` in the interceptor's CloudWatch logs).

### [OKTA]

- No notebook token test — auth is browser-interactive. Launch the app (Step 3) and log in via Okta.
- Log in as the seeded personas (e.g. policyholder001@example.com, policyholder002@example.com, admin) to see identity-scoped results.
- Notes identity flows via RFC 8693 OBO token exchange at GW2.

⏱️ **Heads-up:** the cells above call the live agent/gateways — a few seconds per agent turn; the two FGAC cells make a couple of gateway round-trips per persona.

**Troubleshooting:**

- Check CloudWatch logs for:
  - Agent Runtime logs: `/aws/bedrock-agentcore/runtime/<runtime-id>`
  - Gateway interceptor logs (Cognito path): `/aws/lambda/lakehouse-gateway-interceptor`
- Verify SSM parameters are set correctly
- Ensure all components are deployed in correct order